In [ ]:
import polars as pl

In [ ]:
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re


def aggregate_explainer_parquet(
    logging_dir: str,
    file_patterns: Dict[str, str] = {
        "shap": "*full_explainer_values_test*.parquet",
        "features": "*explainer_rep_test*.parquet",
        "labels": "*explainer_label_test*.parquet",
    },
    experiments: Optional[List[int]] = None,
    folds: Optional[List[int]] = None,
    output_dir: Optional[str] = None,
) -> Dict[str, pl.DataFrame]:
    """
    Aggregate explainer values from parquet files, using directory structure to identify folds.
    """
    logging_path = Path(logging_dir)

    # Debug: Print all found files first
    print("Scanning for files...")
    for data_type, pattern in file_patterns.items():
        files = list(logging_path.rglob(pattern))
        print(f"{data_type}: Found {len(files)} files with pattern '{pattern}'")
        for f in files[:3]:  # Show first 3 files
            print(f"  {f}")

    # Collect files by fold (using directory structure)
    file_groups = {}

    for data_type, pattern in file_patterns.items():
        files = list(logging_path.rglob(pattern))

        if not files:
            print(f"No files found for pattern: {pattern}")
            continue

        for file_path in files:
            try:
                # Extract fold info from directory path
                path_parts = file_path.parts
                fold_num = None

                # Look for fold number in directory path
                for part in path_parts:
                    fold_match = re.search(r"fold[_-]?(\d+)", part, re.IGNORECASE)
                    if fold_match:
                        fold_num = int(fold_match.group(1))
                        break

                # If no fold found in path, use directory name as unique identifier
                if fold_num is None:
                    # Use parent directory name or create sequential fold numbers
                    parent_dir = file_path.parent.name
                    # Try to extract any number from parent directory
                    num_match = re.search(r"(\d+)", parent_dir)
                    if num_match:
                        fold_num = int(num_match.group(1))
                    else:
                        # Use hash of directory path as fold identifier
                        fold_num = hash(str(file_path.parent)) % 1000

                print(
                    f"File: {file_path.name}, Directory: {file_path.parent.name}, Fold: {fold_num}"
                )

                # Filter by fold if specified
                if folds and fold_num not in folds:
                    continue

                # Group files by fold
                if fold_num not in file_groups:
                    file_groups[fold_num] = {}

                # Categorize file type
                if "full_explainer" in pattern or "shap" in data_type.lower():
                    file_groups[fold_num]["shap"] = file_path
                elif "explainer_rep" in pattern:
                    file_groups[fold_num]["features"] = file_path
                elif "label" in pattern:
                    file_groups[fold_num]["labels"] = file_path

            except Exception as e:
                print(f"Error processing {file_path}: {e}")
                continue

    print(f"\nFound {len(file_groups)} fold combinations:")
    for fold_num in sorted(file_groups.keys()):
        available_types = list(file_groups[fold_num].keys())
        print(f"  Fold {fold_num}: {available_types}")

    # Process aligned data
    aligned_data = {"shap_values": [], "feature_values": [], "labels": []}

    feature_columns = None

    for fold_num, files in file_groups.items():
        try:
            # Load all three types for this fold
            fold_data = {}

            for data_type in ["shap", "features", "labels"]:
                if data_type in files:
                    df = pl.read_parquet(files[data_type])

                    # Add metadata
                    df = df.with_columns(
                        [
                            pl.lit(fold_num).alias("fold"),
                        ]
                    )

                    fold_data[data_type] = df

            # Only proceed if we have both SHAP and feature values
            if "shap" in fold_data and "features" in fold_data:
                shap_df = fold_data["shap"]
                features_df = fold_data["features"]
                labels_df = fold_data.get("labels", None)

                # Get feature columns (exclude metadata)
                exclude_cols = ["fold"]
                shap_cols = [col for col in shap_df.columns if col not in exclude_cols]
                feature_cols = [
                    col for col in features_df.columns if col not in exclude_cols
                ]

                # Find common columns
                common_cols = list(set(shap_cols) & set(feature_cols))

                if not common_cols:
                    print(f"No common columns for fold {fold_num}")
                    continue

                # Ensure consistent feature ordering
                if feature_columns is None:
                    feature_columns = sorted(common_cols)
                else:
                    # Keep only features that exist in all folds
                    feature_columns = [
                        col for col in feature_columns if col in common_cols
                    ]

                # Take minimum length to ensure alignment
                min_length = min(len(shap_df), len(features_df))
                if labels_df is not None:
                    min_length = min(min_length, len(labels_df))

                # Slice and reorder columns consistently
                shap_aligned = shap_df.head(min_length).select(
                    feature_columns + ["fold"]
                )

                features_aligned = features_df.head(min_length).select(
                    feature_columns + ["fold"]
                )

                if labels_df is not None:
                    labels_aligned = labels_df.head(min_length).with_columns(
                        [pl.lit(fold_num).alias("fold")]
                    )
                else:
                    # Create dummy labels if not available
                    labels_aligned = pl.DataFrame(
                        {"label": [0] * min_length, "fold": [fold_num] * min_length}
                    )

                aligned_data["shap_values"].append(shap_aligned)
                aligned_data["feature_values"].append(features_aligned)
                aligned_data["labels"].append(labels_aligned)

                print(
                    f"Processed fold {fold_num}: {min_length} samples, {len(feature_columns)} features"
                )

        except Exception as e:
            print(f"Error processing fold {fold_num}: {e}")
            continue

    # Combine all data
    final_data = {}

    for data_type, dfs in aligned_data.items():
        if dfs:
            combined_df = pl.concat(dfs, how="vertical")
            final_data[data_type] = combined_df
        else:
            final_data[data_type] = None

    # Ensure all DataFrames have the same number of rows
    if all(df is not None for df in final_data.values()):
        min_rows = min(len(df) for df in final_data.values())
        for key in final_data:
            final_data[key] = final_data[key].head(min_rows)

    # Save aggregated data
    if output_dir:
        output_path = Path(output_dir)
        output_path.mkdir(exist_ok=True)

        for data_type, df in final_data.items():
            if df is not None and len(df) > 0:
                save_path = output_path / f"aligned_{data_type}.parquet"
                df.write_parquet(save_path)
                print(f"Saved aligned {data_type} to {save_path}")

    # Print summary
    print("\nData Summary:")
    for key, df in final_data.items():
        if df is not None:
            fold_info = df.select(["fold"]).unique().sort(["fold"])
            print(
                f"{key}: {len(df)} rows, {len([col for col in df.columns if col not in ['fold']])} features"
            )
            print(f"  Folds: {len(fold_info)}")

    return final_data

In [ ]:
def plot_shap_summary_heatmap(
    data: Dict[str, pl.DataFrame],
    max_features: int = 20,
    figsize: Tuple[int, int] = (14, 8),
    save_path: Optional[str] = None,
) -> plt.Figure:
    """
    Create a SHAP summary plot with feature values as heatmap background.
    """
    shap_df = data["shap_values"]
    feature_df = data["feature_values"]

    exclude_cols = ["experiment", "fold"]
    feature_cols = [col for col in shap_df.columns if col not in exclude_cols]

    # Convert to numpy
    shap_values = shap_df.select(feature_cols).to_numpy()
    feature_values = feature_df.select(feature_cols).to_numpy()

    # Calculate feature importance and select top features
    feature_importance = np.mean(np.abs(shap_values), axis=0)
    top_indices = np.argsort(feature_importance)[-max_features:]

    top_shap = shap_values[:, top_indices]
    top_features = feature_values[:, top_indices]
    top_names = [feature_cols[i] for i in top_indices]

    # Create figure with subplots
    fig, (ax1, ax2) = plt.subplots(
        1, 2, figsize=figsize, gridspec_kw={"width_ratios": [3, 1]}
    )

    # Main SHAP plot
    n_samples, n_features = top_shap.shape

    for i in range(n_features):
        y_pos = i
        shap_vals = top_shap[:, i]
        feature_vals = top_features[:, i]

        # Normalize feature values for color mapping
        if np.std(feature_vals) > 0:
            feature_norm = (feature_vals - np.min(feature_vals)) / (
                np.max(feature_vals) - np.min(feature_vals)
            )
        else:
            feature_norm = np.zeros_like(feature_vals)

        scatter = ax1.scatter(
            shap_vals,
            [y_pos] * len(shap_vals),
            c=feature_norm,
            cmap="RdYlBu_r",
            alpha=0.6,
            s=15,
            edgecolors="none",
        )

    ax1.set_yticks(range(n_features))
    ax1.set_yticklabels(top_names)
    ax1.set_xlabel("SHAP Value")
    ax1.set_title("SHAP Summary Plot")
    ax1.axvline(x=0, color="black", linestyle="-", alpha=0.3)
    ax1.grid(True, alpha=0.3)

    # Feature importance bar plot
    importance_values = feature_importance[top_indices]
    bars = ax2.barh(range(n_features), importance_values, alpha=0.7, color="steelblue")
    ax2.set_yticks(range(n_features))
    ax2.set_yticklabels([])  # Remove labels since they're on the left plot
    ax2.set_xlabel("Mean |SHAP|")
    ax2.set_title("Feature\nImportance")

    # Add colorbar
    cbar = plt.colorbar(scatter, ax=ax1, fraction=0.046, pad=0.04)
    cbar.set_label("Feature Value\n(Low → High)", rotation=270, labelpad=20)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"Plot saved to {save_path}")

    return fig


def plot_shap_violin_by_feature_value(
    data: Dict[str, pl.DataFrame],
    feature_name: str,
    n_bins: int = 4,
    figsize: Tuple[int, int] = (10, 6),
    save_path: Optional[str] = None,
) -> plt.Figure:
    """
    Create violin plots of SHAP values binned by feature values.
    """
    shap_df = data["shap_values"]
    feature_df = data["feature_values"]

    if feature_name not in shap_df.columns or feature_name not in feature_df.columns:
        raise ValueError(f"Feature '{feature_name}' not found in data")

    # Extract data for the specific feature
    shap_values = shap_df[feature_name].to_numpy()
    feature_values = feature_df[feature_name].to_numpy()

    # Create bins based on feature value quantiles
    bin_edges = np.percentile(feature_values, np.linspace(0, 100, n_bins + 1))
    bin_indices = np.digitize(feature_values, bin_edges) - 1
    bin_indices = np.clip(bin_indices, 0, n_bins - 1)

    # Prepare data for violin plot
    plot_data = []
    bin_labels = []

    for i in range(n_bins):
        mask = bin_indices == i
        if np.sum(mask) > 0:
            bin_shap = shap_values[mask]
            plot_data.extend(bin_shap)
            bin_labels.extend(
                [f"Bin {i + 1}\n[{bin_edges[i]:.2f}, {bin_edges[i + 1]:.2f}]"]
                * len(bin_shap)
            )

    # Create violin plot
    fig, ax = plt.subplots(figsize=figsize)

    df_plot = pl.DataFrame({"SHAP_Value": plot_data, "Feature_Bin": bin_labels})

    # Convert to pandas for seaborn
    df_pandas = df_plot.to_pandas()

    sns.violinplot(data=df_pandas, x="Feature_Bin", y="SHAP_Value", ax=ax)
    ax.set_title(f"SHAP Values Distribution by {feature_name} Bins")
    ax.set_xlabel(f"{feature_name} Value Ranges")
    ax.set_ylabel("SHAP Value")
    ax.axhline(y=0, color="black", linestyle="--", alpha=0.5)
    ax.grid(True, alpha=0.3)

    plt.xticks(rotation=45)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"Plot saved to {save_path}")

    return fig


def plot_feature_interaction_heatmap(
    data: Dict[str, pl.DataFrame],
    top_n: int = 15,
    figsize: Tuple[int, int] = (12, 10),
    save_path: Optional[str] = None,
) -> plt.Figure:
    """
    Create a heatmap showing correlations between SHAP values of different features.
    """
    shap_df = data["shap_values"]
    exclude_cols = ["experiment", "fold"]
    feature_cols = [col for col in shap_df.columns if col not in exclude_cols]

    # Select top features by importance
    shap_values = shap_df.select(feature_cols).to_numpy()
    feature_importance = np.mean(np.abs(shap_values), axis=0)
    top_indices = np.argsort(feature_importance)[-top_n:]

    top_feature_names = [feature_cols[i] for i in top_indices]
    top_shap_values = shap_values[:, top_indices]

    # Calculate correlation matrix
    correlation_matrix = np.corrcoef(top_shap_values.T)

    # Create heatmap
    fig, ax = plt.subplots(figsize=figsize)

    im = ax.imshow(correlation_matrix, cmap="RdBu_r", aspect="auto", vmin=-1, vmax=1)

    # Set ticks and labels
    ax.set_xticks(range(len(top_feature_names)))
    ax.set_yticks(range(len(top_feature_names)))
    ax.set_xticklabels(top_feature_names, rotation=45, ha="right")
    ax.set_yticklabels(top_feature_names)

    # Add correlation values to cells
    for i in range(len(top_feature_names)):
        for j in range(len(top_feature_names)):
            text = ax.text(
                j,
                i,
                f"{correlation_matrix[i, j]:.2f}",
                ha="center",
                va="center",
                color="black" if abs(correlation_matrix[i, j]) < 0.5 else "white",
            )

    ax.set_title("SHAP Value Correlations Between Features")

    # Add colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Correlation Coefficient")

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"Plot saved to {save_path}")

    return fig

In [ ]:
# Quick load and visualize
logging_dir = "/Users/robin/Documents/git/yaib_logs/mimic_demo/CassClassification/XGBClassifier/2025-08-27T11-37-52.510118"
data = aggregate_explainer_parquet(
    logging_dir,
)
# More control over loading
# data = aggregate_explainer_parquet(
#     logging_directory,
#     file_patterns={
#         'shap': '*full_explainer_values_test*.parquet',
#         'features': '*explainer_values_test*.parquet',
#         'labels': '*explainer_label_test*.parquet'
#     },
#     experiments=[0, 1, 2],  # specific experiments
#     output_dir="aggregated_results"
# )
#
# # Custom plots
# fig = plot_shap_summary_from_parquet(
#     data['shap_values'],
#     data['feature_values'],
#     max_features=15,
#     color_palette="plasma",
#     save_path="shap_summary.png"
# )
#
# # Analyze feature stability across folds
# stability = analyze_feature_stability(
#     data['shap_values'],
#     group_by=['experiment', 'fold'],
#     save_path="feature_stability.csv"
# )

In [ ]:
fig1 = plot_shap_summary_heatmap(data, save_path="shap_heatmap.png")

In [ ]:
fig2 = plot_shap_violin_by_feature_value(data, "fio2_max_hist")

In [ ]:
plot_feature_interaction_heatmap(data, save_path="feature_interaction_heatmap.png")

In [ ]:
data

In [ ]:
def plot_shap_violin_summary_heatmap(
    data: Dict[str, pl.DataFrame],
    max_features: int = 20,
    figsize: Tuple[int, int] = (14, 8),
    violin_alpha: float = 0.6,
    point_alpha: float = 0.4,
    point_size: float = 6,
    heatmap_alpha: float = 0.4,
    smoothing_sigma: float = 1.0,
    color_palette: str = "RdYlBu_r",
    save_path: Optional[str] = None,
) -> plt.Figure:
    """
    Create a SHAP summary plot with violin plots, individual points, and feature value heatmap overlay.
    """
    from scipy.ndimage import gaussian_filter

    shap_df = data["shap_values"]
    feature_df = data["feature_values"]

    exclude_cols = ["experiment", "fold"]
    feature_cols = [col for col in shap_df.columns if col not in exclude_cols]

    # Convert to numpy
    shap_values = shap_df.select(feature_cols).to_numpy()
    feature_values = feature_df.select(feature_cols).to_numpy()

    # Calculate feature importance and select top features
    feature_importance = np.mean(np.abs(shap_values), axis=0)
    top_indices = np.argsort(feature_importance)[-max_features:]

    top_shap = shap_values[:, top_indices]
    top_features = feature_values[:, top_indices]
    top_names = [feature_cols[i] for i in top_indices]

    # Create single plot (removed the bar chart that was causing issues)
    fig, ax = plt.subplots(figsize=figsize)

    n_samples, n_features = top_shap.shape

    # Create heatmap background with better scaling
    x_min, x_max = np.percentile(top_shap, [1, 99])  # Use percentiles to avoid outliers
    x_range = x_max - x_min
    x_padding = 0.15 * x_range

    x_grid = np.linspace(x_min - x_padding, x_max + x_padding, 150)
    y_grid = np.linspace(-0.5, n_features - 0.5, n_features * 8)

    # Create feature value heatmap
    heatmap_data = np.zeros((len(y_grid), len(x_grid)))

    for i in range(n_features):
        y_pos = i
        shap_vals = top_shap[:, i]
        feature_vals = top_features[:, i]

        # Normalize feature values
        if np.std(feature_vals) > 0:
            feature_norm = (feature_vals - np.min(feature_vals)) / (
                np.max(feature_vals) - np.min(feature_vals)
            )
        else:
            feature_norm = np.zeros_like(feature_vals)

        # Create density map with better weighting
        for shap_val, feat_val in zip(shap_vals, feature_norm):
            x_idx = np.argmin(np.abs(x_grid - shap_val))
            y_idx_center = np.argmin(np.abs(y_grid - y_pos))

            # Add Gaussian blob with better scaling
            sigma_x = 0.02 * x_range
            sigma_y = 0.15

            y_start = max(0, y_idx_center - 15)
            y_end = min(len(y_grid), y_idx_center + 16)
            x_start = max(0, x_idx - 8)
            x_end = min(len(x_grid), x_idx + 9)

            for yi in range(y_start, y_end):
                for xi in range(x_start, x_end):
                    weight = np.exp(
                        -(
                            (y_grid[yi] - y_pos) ** 2 / (2 * sigma_y**2)
                            + (x_grid[xi] - shap_val) ** 2 / (2 * sigma_x**2)
                        )
                    )
                    heatmap_data[yi, xi] += feat_val * weight

    # Apply smoothing
    if smoothing_sigma > 0:
        heatmap_data = gaussian_filter(heatmap_data, sigma=smoothing_sigma)

    # Plot heatmap background
    im = ax.imshow(
        heatmap_data,
        extent=[x_grid[0], x_grid[-1], -0.5, n_features - 0.5],
        aspect="auto",
        origin="lower",
        cmap=color_palette,
        alpha=heatmap_alpha,
        interpolation="bilinear",
    )

    # Create violin plots with better positioning
    for i in range(n_features):
        y_pos = i
        shap_vals = top_shap[:, i]

        if len(shap_vals) > 1:
            violin = ax.violinplot(
                [shap_vals],
                positions=[y_pos],
                widths=0.7,
                showmeans=False,
                showmedians=True,
                showextrema=False,
                vert=False,
            )  # ADD THIS LINE - make violins horizontal

            # Customize violin appearance
            for pc in violin["bodies"]:
                pc.set_facecolor("white")  # Change from 'none' to 'white'
                pc.set_edgecolor("darkblue")
                pc.set_alpha(violin_alpha)  # Increase alpha
                pc.set_linewidth(1)
                pc.set_zorder(3)  # ADD THIS - put violins on top

            # Customize median line
            if "cmedians" in violin:
                violin["cmedians"].set_colors("red")
                violin["cmedians"].set_linewidth(3)
                violin["cmedians"].set_zorder(4)  # ADD THIS - put median on top
    # Overlay individual points with better jittering
    scatter_norm = plt.Normalize(vmin=0, vmax=1)

    for i in range(n_features):
        y_pos = i
        shap_vals = top_shap[:, i]
        feature_vals = top_features[:, i]

        # Normalize feature values for coloring
        if np.std(feature_vals) > 0:
            feature_norm = (feature_vals - np.min(feature_vals)) / (
                np.max(feature_vals) - np.min(feature_vals)
            )
        else:
            feature_norm = np.zeros_like(feature_vals)

        # Add jitter with better scaling
        y_jitter = np.random.normal(y_pos, 0.08, len(shap_vals))

        scatter = ax.scatter(
            shap_vals,
            y_jitter,
            c=feature_norm,
            cmap=color_palette,
            alpha=point_alpha,
            s=point_size,
            edgecolors="none",
            rasterized=True,
            norm=scatter_norm,
        )

    # Add simplified legend
    # Customize plot - ADD THESE LINES
    ax.set_yticks(range(n_features))
    ax.set_yticklabels(top_names, fontsize=10)
    ax.set_xlabel("SHAP Value", fontsize=12)
    ax.set_title(
        "SHAP Summary Plot with Violin Distribution & Feature Value Heatmap",
        fontsize=14,
    )
    ax.axvline(x=0, color="black", linestyle="-", alpha=0.8, linewidth=1.5)
    ax.grid(True, alpha=0.3, axis="x")
    ax.set_ylim(-0.5, n_features - 0.5)

    # Set better x-axis limits
    ax.set_xlim(x_min - x_padding, x_max + x_padding)

    # Add single colorbar for feature values
    cbar = plt.colorbar(scatter, ax=ax, fraction=0.046, pad=0.04, shrink=0.8)
    cbar.set_label("Feature Value (Low → High)", rotation=270, labelpad=20, fontsize=10)
    cbar.ax.tick_params(labelsize=8)
    legend_elements = [
        plt.Line2D(
            [0],
            [0],
            color="black",
            linewidth=1.5,
            alpha=violin_alpha,
            label="SHAP Distribution",
        ),
        plt.Line2D([0], [0], color="red", linewidth=2, label="Median SHAP"),
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="gray",
            markersize=4,
            alpha=point_alpha,
            label="Individual Points",
        ),
    ]
    ax.legend(
        handles=legend_elements,
        loc="upper right",
        bbox_to_anchor=(0.98, 0.98),
        fontsize=9,
        framealpha=0.9,
    )

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
        print(f"Plot saved to {save_path}")

    return fig

In [ ]:
fig = plot_shap_violin_summary_heatmap(
    data, max_features=15, violin_alpha=0.3, heatmap_alpha=0, point_alpha=0.7
)